# 04 — Advanced Variants and Extensions of Physics-Informed Neural Networks

> *"To progress in physics is to discover the next decimal — and to invent the next abstraction."* — paraphrasing R. Feynman

The vanilla PINN of Notebook 1 is conceptually elegant but fragile. As surveyed in Notebook 3, it fails for stiff PDEs, complex geometries, sharp features, and uncertainty-aware inverse problems. The post-2019 literature responded with a family of *structurally redesigned* PINNs, each tailored to one weakness of the original.

This notebook treats four families with the analytical rigor of mathematical physics and statistics:

1. **Conservative PINNs (cPINNs)** — enforcing exact conservation laws and flux continuity by reformulating the residual at the level of fluxes;
2. **Variational PINNs (VPINNs)** — replacing the strong-form residual by a Petrov–Galerkin weak form, recovering the test-function machinery of FEM;
3. **Extended PINNs (XPINNs)** — decomposing $\Omega\times[0,T]$ into subdomains and stitching local PINNs across interfaces;
4. **Bayesian PINNs (B-PINNs)** — turning $\theta$ into a random variable and propagating uncertainty via Hamiltonian Monte Carlo or variational inference.

Each family is presented as a *principled* modification: we derive the new loss functional from physical or statistical first principles, study its theoretical properties, and discuss when and why to use it.

## Part I — Conservative PINNs (cPINNs)

### I.1 Motivation: PDEs as conservation laws

Many PDEs of mathematical physics — Euler, Navier–Stokes, Maxwell, magnetohydrodynamics, transport equations — are *conservation laws* in the strict sense:

$$
\partial_t \mathbf{u}(x,t) + \nabla\cdot \mathbf{F}(\mathbf{u},\nabla\mathbf{u},x,t) \;=\; \mathbf{S}(\mathbf{u},x,t),
$$

where $\mathbf{u}$ is the conserved quantity (density, momentum, energy, charge), $\mathbf{F}$ the flux tensor, and $\mathbf{S}$ a source. Integrating over an arbitrary control volume $V \subset \Omega$ and using the divergence theorem yields the integral form

$$
\frac{d}{dt}\int_V \mathbf{u}\,dV + \oint_{\partial V} \mathbf{F}\cdot\mathbf{n}\,dS \;=\; \int_V \mathbf{S}\,dV.
$$

**Vanilla PINNs penalize the differential residual pointwise — they do not enforce this integral identity exactly.** Consequently $\partial_t \int_V u\,dV$ generally drifts: vanilla PINN solutions *do not conserve mass*. For hyperbolic conservation laws with shocks, this is fatal: Rankine–Hugoniot jump conditions cannot be captured.

### I.2 The cPINN ansatz (Jagtap, Kharazmi, Karniadakis 2020)

Decompose $\Omega$ into non-overlapping subdomains $\Omega = \bigsqcup_{k=1}^K \Omega_k$ with shared interfaces $\Gamma_{kl} = \overline{\Omega_k}\cap\overline{\Omega_l}$. Assign each subdomain its own neural network $u^{(k)}_{\theta_k}$ trained on its own PDE residual. Enforce **flux continuity** explicitly at interfaces:

$$
\big[\!\big[\,\mathbf{F}\!\cdot\!\mathbf{n}\,\big]\!\big]_{\Gamma_{kl}} = \mathbf{F}(u^{(k)})\cdot\mathbf{n} - \mathbf{F}(u^{(l)})\cdot\mathbf{n} \;\stackrel{!}{=}\; 0,
$$

together with continuity of the average solution

$$
\big\{\!\big\{ u \big\}\!\big\}_{\Gamma_{kl}} = \tfrac{1}{2}(u^{(k)} + u^{(l)}) - \overline{u}_{\Gamma_{kl}} \;\stackrel{!}{=}\; 0.
$$

The cPINN loss is

$$
\mathcal{L}_{\text{cPINN}} \;=\; \sum_{k=1}^K \big(\lambda_r \mathcal{L}_r^{(k)} + \lambda_b \mathcal{L}_b^{(k)} + \lambda_0 \mathcal{L}_0^{(k)}\big) + \lambda_F \sum_{(k,l)} \mathcal{L}_F^{(kl)} + \lambda_u \sum_{(k,l)} \mathcal{L}_u^{(kl)},
$$

where $\mathcal{L}_F^{(kl)} = \sum_i \| [\![\mathbf{F}\cdot\mathbf{n}]\!](x^{kl}_i,t^{kl}_i)\|^2$ and $\mathcal{L}_u^{(kl)}$ analogous for the average jump.

### I.3 Properties

* **Local resolution.** Each $u^{(k)}$ can use a different network size, frequency embedding, or activation, adapted to the local regularity of the solution. Useful when the solution has localized features (shocks, vortices).
* **Parallelism.** Subdomain PINNs train almost independently; only the interface loss couples them. Excellent multi-GPU scaling.
* **Exact flux continuity.** As $\mathcal{L}_F^{(kl)} \to 0$, the integral conservation law is satisfied across interfaces by design.
* **Hyperbolic-friendly.** For weak solutions of conservation laws, the Rankine–Hugoniot jump conditions $[\![\mathbf{F}]\!] = s\,[\![\mathbf{u}]\!]$ (with shock speed $s$) become explicit interface constraints.

### I.4 Theoretical underpinning — entropy and weak solutions

For genuinely nonlinear hyperbolic problems classical solutions exist only for a finite time after which shocks form. Weak solutions to $\partial_t u + \partial_x F(u) = 0$ exist but are non-unique; the *entropy condition* selects the physically admissible one. A cPINN can incorporate entropy-pair residuals

$$
\partial_t \eta(u_\theta) + \partial_x q(u_\theta) \;\le\; 0\quad\text{(in the weak sense)}
$$

as an additional inequality penalty $\big[\max(0, \partial_t \eta + \partial_x q)\big]^2$. This yields *entropy-stable PINNs* (Patel et al. 2022).

## Part II — Variational PINNs (VPINNs)

### II.1 Motivation: the loss of regularity in strong forms

The vanilla PINN demands $u_\theta$ to be smooth enough that $\mathcal{N}[u_\theta]$ is pointwise defined — at minimum, $u_\theta \in C^m(\Omega)$ for an $m$-th order operator. This excludes solutions with low Sobolev regularity (shocks, point singularities, Dirac source terms) which are commonplace in physics.

FEM avoids this by transferring derivatives onto a test function via integration by parts (Notebook 1, §3). VPINNs (Kharazmi–Zhang–Karniadakis 2019, 2021) borrow this machinery directly.

### II.2 The Petrov–Galerkin weak form

Consider a PDE $\mathcal{N}[u] = f$ on $\Omega$ with $\mathcal{N}$ a second-order elliptic operator $\mathcal{N}[u] = -\nabla\cdot(\kappa(x)\nabla u) + c(x) u$. Choose a *test space* $\mathcal{V} \subset H^1_0(\Omega)$ and pose

$$
\int_\Omega \big(\kappa\nabla u_\theta\cdot \nabla v + c u_\theta v\big)\,dx \;=\; \int_\Omega f\,v\,dx,\qquad \forall v\in\mathcal{V}.
$$

The derivative of $u$ now appears in $H^1$ only — $u_\theta$ need only be $H^1$, not $C^2$.

### II.3 The VPINN loss

Choose a finite collection of test functions $\{v_j\}_{j=1}^J \subset \mathcal{V}$ (e.g. Legendre polynomials of degree up to $J$, sine modes, or piecewise polynomials on a mesh). Define *variational residuals*

$$
R_j(\theta) \;:=\; \int_\Omega \big(\kappa\nabla u_\theta\cdot \nabla v_j + c u_\theta v_j - f v_j\big)\,dx.
$$

The VPINN loss is

$$
\mathcal{L}_{\text{VPINN}}(\theta) \;=\; \sum_{j=1}^J w_j\,R_j(\theta)^2 \;+\; \text{(BC, IC, data terms)}.
$$

### II.4 Computing the integrals

Each $R_j$ is an integral over $\Omega$ approximated by quadrature — Gauss–Legendre, Gauss–Lobatto, or Monte Carlo with $N_q$ quadrature points. The derivative on $u_\theta$ is one order *lower* than in the strong-form PINN — a major win for regularity.

### II.5 hp-VPINNs (Kharazmi et al. 2021)

Generalize by partitioning $\Omega$ into elements $\{K_e\}$ (the *h*-refinement) and assigning each element local polynomial test functions of degree $p$. Both element size $h$ and polynomial degree $p$ are tuneable — a *spectral element* version of VPINN with proven exponential convergence for analytic solutions and algebraic convergence for those with isolated singularities.

### II.6 Trade-offs versus strong-form PINN

* **(+)** Lower regularity requirements on $u_\theta$ — captures discontinuous-derivative solutions.
* **(+)** Better-conditioned NTK in many cases, because the differential operator is shifted onto deterministic test functions.
* **(+)** Natural treatment of Neumann boundary conditions through integration by parts.
* **(–)** Quadrature cost: each $R_j$ requires $N_q$ network evaluations.
* **(–)** Choice of test space introduces an extra design dimension.
* **(–)** Nonlinear PDEs do not admit a clean bilinear form; loss becomes more complex.

### II.7 Connection to Deep Ritz Method

When the PDE is the Euler–Lagrange equation of an energy functional $E[u]$ (as in Poisson, linear elasticity, hyperelasticity), one can simply minimize $E[u_\theta]$ — the *Deep Ritz Method* (E & Yu 2018). This is a special case of VPINN with $u_\theta$ as its own test function, valid only in the variational regime.

## Part III — Extended PINNs (XPINNs) and Domain Decomposition

### III.1 Motivation: scaling to large or geometrically complex domains

A monolithic PINN struggles when:

* $\Omega$ is geometrically complex (multiply connected, thin boundary layers, embedded interfaces);
* solution regularity varies across the domain (smooth in some regions, oscillatory in others);
* the spatial extent of $\Omega$ requires more network capacity than a single MLP can deliver.

Domain decomposition methods have a long history in FEM (Schwarz iterations, BDDC, FETI). XPINNs (Jagtap–Karniadakis 2020) port the concept to PINNs.

### III.2 The XPINN construction

Partition $\Omega\times(0,T) = \bigcup_{k=1}^K \Omega_k$ into space–time subdomains (possibly overlapping or non-overlapping). Assign each subdomain its own network $u^{(k)}_{\theta_k}$. Interface continuity is enforced via two penalties:

* **Solution continuity:** $u^{(k)} = u^{(l)}$ on $\Gamma_{kl}$;
* **Residual continuity:** $\mathcal{N}[u^{(k)}] = \mathcal{N}[u^{(l)}]$ on $\Gamma_{kl}$ — i.e. *the PDE itself agrees from both sides*.

This second condition, unique to XPINNs and absent from classical FEM domain decomposition, makes use of the *operator-level* information available in PINN: we can demand not only $C^0$ matching of $u$ but $C^m$ matching at the level of the residual.

### III.3 The XPINN loss

$$
\mathcal{L}_{\text{XPINN}} \;=\; \sum_{k=1}^K \mathcal{L}^{(k)}_{\text{PINN}}(\theta_k) \;+\; \lambda_I \sum_{(k,l)\,\text{interfaces}}\Big(\mathcal{L}_{u,kl} + \mathcal{L}_{r,kl}\Big),
$$

with $\mathcal{L}_{u,kl} = \sum_i |u^{(k)} - u^{(l)}|^2|_{\Gamma_{kl}}$ and $\mathcal{L}_{r,kl} = \sum_i |\mathcal{N}[u^{(k)}] - \mathcal{N}[u^{(l)}]|^2|_{\Gamma_{kl}}$.

### III.4 Why this helps

Each $u^{(k)}$ sees a smaller, simpler problem; the NTK spectrum within $\Omega_k$ is less ill-conditioned. Spectral bias is mitigated locally — a small subdomain with smooth solution converges quickly, while a different subdomain hosting a shock can use a higher-frequency Fourier feature embedding.

### III.5 cPINN vs XPINN — a clean distinction

Both decompose the domain, but the interface coupling differs:

| Aspect | cPINN | XPINN |
|---|---|---|
| Interface variable | Flux $\mathbf{F}\cdot\mathbf{n}$ | $u$ and $\mathcal{N}[u]$ |
| PDE type | Conservation law | Generic |
| Continuity order | Flux ($C^{-1}$ in $u$) | Solution + residual ($C^m$) |
| Best for | Shock-capturing, mass conservation | Geometric / regularity decomposition |

### III.6 Convergence theory

Hu–Jagtap–Karniadakis (2022) prove that with sufficient interface penalty weight, the XPINN solution converges to the global PDE solution as subdomain sizes shrink and network capacities grow. The penalty plays the role of a *Schwarz alternation* enforcer in the limit.

## Part IV — Bayesian PINNs (B-PINNs)

### IV.1 Motivation: uncertainty as a scientific necessity

Vanilla PINNs return a *point estimate* $u_{\theta^\ast}$. In scientific applications — climate, biomechanics, astrophysics — point estimates are inadequate. We want a *posterior distribution* over $u$ that reflects:

* measurement noise in the data $\mathcal{L}_d$;
* model misspecification (e.g. uncertain coefficients in $\mathcal{N}$);
* finite-sample variance in collocation point sets.

The natural language is Bayesian.

### IV.2 The Bayesian model

Let $\theta \in \mathbb{R}^p$ with prior $\theta \sim p(\theta)$. Define a likelihood through the PINN loss components:

* Data likelihood: $u^d_i = u_\theta(x^d_i,t^d_i) + \varepsilon^d_i$, $\varepsilon^d_i \sim \mathcal{N}(0,\sigma_d^2)$.
* PDE likelihood: $\mathcal{N}[u_\theta](x^r_i,t^r_i) - f(x^r_i,t^r_i) = \varepsilon^r_i$, $\varepsilon^r_i \sim \mathcal{N}(0,\sigma_r^2)$.
* Similarly for BC, IC.

The joint log-posterior is

$$
\log p(\theta \mid \mathcal{D}) \;=\; \log p(\theta) \;-\; \frac{1}{2\sigma_r^2}\sum_i \big(\mathcal{N}[u_\theta]-f\big)_i^2 \;-\; \frac{1}{2\sigma_d^2}\sum_i \big(u_\theta - u^d\big)_i^2 \;-\; \cdots \;+\; \text{const.}
$$

This is precisely the PINN loss with $\lambda$'s identified with inverse-variance precisions, plus a prior regularizer — *the deterministic PINN is the maximum a posteriori (MAP) estimate of a B-PINN*.

### IV.3 Hamiltonian Monte Carlo for B-PINNs (Yang–Meng–Karniadakis 2021)

Sample $\theta$ from the posterior using HMC. Augment the state with a momentum $p$ and a Hamiltonian

$$
H(\theta,p) \;=\; -\log p(\theta\mid\mathcal{D}) \;+\; \tfrac{1}{2}p^\top M^{-1} p.
$$

Simulate Hamilton's equations for $L$ leapfrog steps and accept/reject via Metropolis. Cost per HMC step: $\mathcal{O}(L)$ PINN loss evaluations plus their gradients — expensive but exact.

Posterior mean and credible intervals are then estimated as Monte Carlo averages:

$$
\mathbb{E}[u(x,t)\mid \mathcal{D}] \approx \frac{1}{S}\sum_{s=1}^S u_{\theta^{(s)}}(x,t),\qquad \mathbb{V}[u(x,t)\mid \mathcal{D}] \approx \frac{1}{S}\sum_{s=1}^S \big(u_{\theta^{(s)}}-\bar u\big)^2.
$$

### IV.4 Variational Inference

When HMC is too expensive, fit a variational posterior $q_\phi(\theta) \approx p(\theta\mid\mathcal{D})$ (typically mean-field Gaussian, $q_\phi(\theta) = \prod_i \mathcal{N}(\theta_i;\mu_i,\sigma_i^2)$) by maximizing the evidence lower bound (ELBO):

$$
\mathcal{L}_{\text{ELBO}}(\phi) \;=\; \mathbb{E}_{\theta\sim q_\phi}\big[\log p(\mathcal{D}\mid\theta)\big] - \mathrm{KL}\big(q_\phi \,\|\, p(\theta)\big).
$$

Reparameterize $\theta = \mu + \sigma \odot \xi$ with $\xi \sim \mathcal{N}(0,I)$ to enable gradient-based optimization (the Kingma–Welling reparameterization trick). The trained $q_\phi$ supplies cheap posterior samples.

### IV.5 Stein Variational Gradient Descent

SVGD (Liu–Wang 2016) maintains a set of $M$ particles $\{\theta_i\}$ deterministically updated by

$$
\theta_i \leftarrow \theta_i + \eta\,\phi^\ast(\theta_i),\qquad \phi^\ast(\theta) = \frac{1}{M}\sum_j \big[ k(\theta_j,\theta)\nabla_{\theta_j}\log p(\theta_j\mid\mathcal{D}) + \nabla_{\theta_j} k(\theta_j,\theta)\big].
$$

with kernel $k$. Particles converge in distribution to the posterior; no mode collapse, no random sampling. Used in PINNs with success for moderate-dimensional inverse problems.

### IV.6 What B-PINNs deliver

* **Predictive credible intervals** $u(x,t) \in [\hat u(x,t) - 2\hat\sigma, \hat u(x,t) + 2\hat\sigma]$ at any $(x,t)$.
* **Posterior on physical parameters** in inverse problems: $p(\nu \mid \mathcal{D})$ for viscosity, etc.
* **Active learning loops**: pick the next measurement at the point of maximum predictive variance.
* **Robust failure detection**: high posterior variance flags regions where the PDE is poorly constrained.

### IV.7 Limitations

* HMC requires $\mathcal{O}(10^4)$ samples for $\mathcal{O}(10^4)$ parameters — heroic for million-parameter PINNs.
* VI underestimates posterior variance (mean-field Gaussian is overly tight).
* Selecting noise priors $\sigma_r, \sigma_d$ is non-trivial — they couple to the inferred uncertainty in nontrivial ways.

## Part V — A Brief Tour of Other Variants

For completeness we mention several other influential PINN extensions, with shorter treatments.

### V.1 Stochastic PINNs (sPINNs)

For SPDEs $du = \mathcal{N}[u]\,dt + g(u)\,dW$ with Wiener noise $W$, treat the *characteristic functional* of the solution by Wiener chaos expansion: $u(x,t,\omega) = \sum_\alpha c_\alpha(x,t) H_\alpha(\omega)$ with Hermite polynomials $H_\alpha$. PINN approximates each $c_\alpha$ separately. Equivalent at the level of moments to a polynomial-chaos / generalized polynomial chaos PINN.

### V.2 Operator-learning PINNs (DeepONet, FNO + physics)

Instead of solving one PDE, *learn the solution operator* $\mathcal{G} : f \mapsto u$. DeepONet (Lu–Jin–Karniadakis 2021) and FNO (Li et al. 2020) are surrogate operator approximators; physics-informed extensions (PI-DeepONet, PI-FNO) incorporate the PDE residual as a regularizer. The resulting maps amortize over an entire family of input functions $f$ — a single offline training, then $\mathcal{O}(1)$ evaluation per problem instance.

### V.3 Self-adaptive PINNs (SA-PINN)

McClenny–Braga-Neto (2020): attach per-point trainable weights $\lambda_i$ to each collocation point and *maximize* over them while minimizing over $\theta$:

$$
\min_\theta \max_{\lambda \ge 0}\;\frac{1}{N_r}\sum_i \lambda_i\,|r_\theta(x_i,t_i)|^2 + (\text{soft constraint on}\;\lambda).
$$

Game-theoretic, reminiscent of GAN training. Achieves automatic point-wise reweighting and shock detection.

### V.4 Hamiltonian / symplectic PINNs

For Hamiltonian PDEs (Schrödinger, KdV) build the structure of the Poisson bracket into the network architecture itself. Greydanus–Dzamba–Yosinski 2019 (HNN); applied to PDEs in subsequent work. Energy is conserved by construction.

### V.5 Lagrangian PINNs

Track Lagrangian trajectories $X(t)$ instead of Eulerian fields $u(x,t)$. Useful for advection-dominated flows.

### V.6 Curriculum / sequence-to-sequence PINNs

Solve a *family* of PDEs of increasing difficulty (smaller-Reynolds first, then larger; longer time horizon progressively). Each network warm-starts the next.

## Part VI — Choosing the Right Variant

The decision tree below condenses field experience.

| Symptom of the problem | Recommended variant |
|---|---|
| Conservation laws with shocks | cPINN (+ entropy regularizer) |
| Low-regularity solutions, Neumann BCs | VPINN / hp-VPINN / Deep Ritz |
| Large domain or heterogeneous regularity | XPINN |
| Noisy data, need credible intervals | B-PINN (HMC if affordable, else VI) |
| Stochastic source term or coefficient | sPINN / gPC-PINN |
| Same PDE family with many parameter values | PI-DeepONet / PI-FNO |
| Sharp features but no shocks | Fourier-feature PINN + RAR sampling |
| Multi-physics coupling | XPINN with subdomain-specific operators |

These are *families*, not silver bullets — most production PINN pipelines stack several techniques (e.g. XPINN with NTK weighting and Fourier features inside each subdomain). The unifying theme is that the vanilla PINN is the starting point, not the destination.

## Part VII — Outlook

The advanced variants share a common philosophy: *encode more physics into the architecture or loss*, leaving less work for the optimizer. cPINN encodes conservation; VPINN encodes variational structure; XPINN encodes geometric locality; B-PINN encodes probabilistic structure.

Looking forward, three active research threads stand out.

1. **Operator learning at scale.** PI-FNO/DeepONet and follow-on Transformer-based neural operators (e.g. UPT, GNOT) promise *amortized PDE solvers* trained once, queried billions of times — relevant for climate emulators and digital twins.
2. **Provable certification.** Bridging PINNs to rigorous a-posteriori error estimators (à la FEM) is an open problem; current bounds are loose by 1–2 orders of magnitude.
3. **Foundation models for physics.** Large pretrained models on diverse PDE corpora, fine-tuned per problem with a small PINN-style residual loss — the analog of GPT for partial differential equations.

In the next and final theory notebook we pivot from generality to a single concrete equation — the 1-D viscous Burgers' equation — that has become the *hello world* of PINNs. Its mix of nonlinear advection, viscous diffusion, and incipient shock formation exercises every facet of theory developed so far.

## References

1. **Jagtap, A. D., Kharazmi, E., Karniadakis, G. E.** *Conservative physics-informed neural networks on discrete domains for conservation laws: Applications to forward and inverse problems.* CMAME 365 (2020).
2. **Jagtap, A. D., Karniadakis, G. E.** *Extended physics-informed neural networks (XPINNs): A generalized space-time domain decomposition based deep learning framework for nonlinear partial differential equations.* CICP 28 (2020).
3. **Kharazmi, E., Zhang, Z., Karniadakis, G. E.** *Variational physics-informed neural networks for solving partial differential equations.* arXiv:1912.00873 (2019).
4. **Kharazmi, E., Zhang, Z., Karniadakis, G. E.** *hp-VPINNs: Variational physics-informed neural networks with domain decomposition.* CMAME 374 (2021).
5. **E, W., Yu, B.** *The Deep Ritz method.* CMS 6 (2018).
6. **Yang, L., Meng, X., Karniadakis, G. E.** *B-PINNs: Bayesian physics-informed neural networks for forward and inverse PDE problems with noisy data.* J. Comp. Phys. 425 (2021).
7. **Liu, Q., Wang, D.** *Stein variational gradient descent.* NeurIPS 2016.
8. **Lu, L., Jin, P., Karniadakis, G. E.** *DeepONet: Learning nonlinear operators for identifying differential equations.* Nature MI 3 (2021).
9. **Li, Z. et al.** *Fourier neural operator for parametric partial differential equations.* ICLR 2021.
10. **McClenny, L., Braga-Neto, U.** *Self-adaptive physics-informed neural networks using a soft attention mechanism.* arXiv:2009.04544 (2020).
11. **Patel, R., Manickam, I., Trask, N., Wood, M.** *Thermodynamically consistent physics-informed neural networks for hyperbolic systems.* J. Comp. Phys. 449 (2022).
12. **Hu, Z., Jagtap, A. D., Karniadakis, G. E.** *When do extended physics-informed neural networks (XPINNs) improve generalization?* SIAM J. Sci. Comput. 44 (2022).